# Replication: the authors' open models on a stratified sample

Runs GOT-OCR2, DeepSeek-VL2-tiny, Phi-3.5-Vision, Qwen2.5-VL-7B, InternVL3-8B, Pixtral-12B, TrOCR and PaddleOCR on the
stratified Turkish sample, each in its own environment, and compares the predictions with the authors'. GPT-4o and
Claude are left out as commercial.

**Runtime → Change runtime type:** A100 runs all eight. L4 (22 GB) skips Pixtral. T4 (15 GB) also skips Qwen and
InternVL.

**Before running:** `colab/make_bundle.sh`, then put `colab/ocr_bundle.zip` in `MyDrive/ocr-benchmark/` (or upload it
to `/content`). Results go to `MyDrive/ocr-benchmark/replication/`. After a disconnect use Run all again: finished
models are skipped, an interrupted one continues.


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!df -h /content | tail -1

from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive/ocr-benchmark")
OUT = DRIVE / "replication"
OUT.mkdir(parents=True, exist_ok=True)
print("results:", OUT)


In [ ]:
# Unpack the code and the sample
import shutil, zipfile

bundle = next((p for p in (DRIVE / "ocr_bundle.zip", Path("/content/ocr_bundle.zip")) if p.exists()), None)
assert bundle, "ocr_bundle.zip not found in MyDrive/ocr-benchmark or /content"
shutil.rmtree("/content/ocr", ignore_errors=True)
zipfile.ZipFile(bundle).extractall("/content/ocr")
%cd /content/ocr
!pip install -q uv jiwer editdistance
!ls replication/data/syntetic-turkish-dataset/benchmark/text_category/*/*.png | wc -l


Optional: check all models on 3 images each first (about 30-40 min with the downloads) by adding `--smoke-only`.
A full run does the same check before each model anyway, so a broken model fails early and the next one starts.


In [ ]:
!python colab/run_replication.py --out "{OUT}"


In [ ]:
# Compare with the authors' predictions; also works on partial results
import glob, pandas as pd

for path in glob.glob(f"{OUT}/*.csv"):
    shutil.copy(path, "replication/results/")
!python replication/analyze.py
for path in glob.glob("replication/results/*vs_authors.csv"):
    shutil.copy(path, OUT)
print(open(OUT / "runs.json").read())
pd.read_csv("replication/results/rerun_vs_authors.csv").query("type == 'all'")
